# Held-out test evaluation of the CUB-200 rerankers (JupyterHub)

Runs the full test pipeline end to end:

1. Sync the repo and install it.
2. Configure Kaggle credentials (prompted; nothing is stored in this notebook).
3. Download the image split, SigLIP embeddings and the M=100 teacher artifact.
   CLIP embeddings and CUB images are fetched automatically by the scripts.
4. Train M=30/50/100 rerankers with one shared recipe (the sweep-best M=30 settings).
5. Compare validation accuracy and **fix the primary selector before looking at test results**.
6. Smoke test, then the full test evaluation (resumable), then a results summary.

All code changes must be pushed to `origin/main` first; cell 2 pulls them.
Re-running a training or evaluation cell skips finished work.

In [ ]:
# --- Settings: edit these, then run all cells top to bottom. ---
import os
from pathlib import Path

REPO_DIR = Path("/home/jovyan/shared/jd.tan/icl-retr-multimodal/icl-retr-multimodal")
ARTIFACT_ROOT = Path("/home/jovyan/shared/jd.tan/icl-retr-multimodal/icl-artifacts")

# Teacher data the rerankers are trained on; also verified against at test time.
TEACHER_DATASET = "juandatan/cub-200-reranker-teacher-data-m100"
TEACHER_ARTIFACT = ARTIFACT_ROOT / "cub-200-reranker-teacher-data-m100" / "reranker_teacher_data.pkl"

# Candidate-pool sizes to train and evaluate. Each maps to selectors.reranker_m{M}
# in configs/eval_reranker_test.yaml, so only 30, 50 and 100 are valid.
POOL_SIZES = [30, 50, 100]

# Headline selector. "auto" takes the highest validation accuracy from the
# validation table (ties go to the smaller pool), so it is fixed before any
# test query is scored. Or name one explicitly, e.g. "reranker_m50".
# Never change it after seeing test results.
PRIMARY_SELECTOR = "auto"

# Kaggle dataset for (partial) test results; None keeps results local only.
RESULTS_KAGGLE_DATASET = "juandatan/cub-200-reranker-test-eval"

# "auto" resumes from the newest partial result; None always starts fresh;
# or give an explicit .pkl path.
RESUME_FROM = "auto"

TRAIN_OUTPUT_ROOT = REPO_DIR / "outputs" / "reranker_test_runs"
EVAL_OUTPUT_ROOT = REPO_DIR / "outputs" / "evals"
SMOKE_OUTPUT_ROOT = REPO_DIR / "outputs" / "evals_smoke"

os.environ["ARTIFACT_ROOT"] = str(ARTIFACT_ROOT)
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["TMPDIR"] = str(ARTIFACT_ROOT / "huggingface-tmp")
Path(os.environ["TMPDIR"]).mkdir(parents=True, exist_ok=True)

In [ ]:
# Streams a command's output into the cell and fails loudly on a non-zero exit.
import shlex
import subprocess
import sys


def run(*args, cwd=None):
    args = [str(arg) for arg in args]
    print("$", " ".join(shlex.quote(arg) for arg in args), flush=True)
    process = subprocess.Popen(
        args, cwd=cwd or REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait() != 0:
        raise RuntimeError(f"Command failed with exit code {process.returncode}: {args[:4]}")


if not REPO_DIR.exists():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    run("git", "clone", "https://github.com/juandatan/icl-retr-multimodal.git", REPO_DIR, cwd=REPO_DIR.parent)
# Fast-forward only, so local edits on the hub are never silently discarded.
run("git", "pull", "--ff-only", "origin", "main")
run("git", "log", "-1", "--oneline")
os.chdir(REPO_DIR)

In [ ]:
%pip install --no-cache-dir --force-reinstall --no-deps "torchvision==0.23.0" --index-url https://download.pytorch.org/whl/cu128
%pip install --upgrade-strategy only-if-needed -e ".[awq]"

In [ ]:
# Kaggle credentials are entered interactively and kept in this kernel's
# environment only, which the scripts' Kaggle CLI calls inherit.
import getpass

if (Path.home() / ".kaggle" / "kaggle.json").exists():
    print("Using existing ~/.kaggle/kaggle.json")
elif not (os.environ.get("KAGGLE_API_TOKEN") or os.environ.get("KAGGLE_KEY")):
    secret = getpass.getpass("Kaggle API token or legacy key: ").strip()
    if secret.startswith("KGAT_"):
        os.environ["KAGGLE_API_TOKEN"] = secret
    else:
        os.environ["KAGGLE_USERNAME"] = input("Kaggle username: ").strip()
        os.environ["KAGGLE_KEY"] = secret
    del secret
run("kaggle", "datasets", "list", "--mine", "--max-size", "1")

In [ ]:
# Inputs not fetched automatically by the scripts. Downloads are skipped when
# the files already exist.
SPLIT_DIR = ARTIFACT_ROOT / "cub-200"
SIGLIP_DIR = ARTIFACT_ROOT / "embeddings" / "cub-200-siglip-embeddings-distractor-set"
DOWNLOADS = [
    ("juandatan/cub-200-image-split", SPLIT_DIR, "image_split.json"),
    ("juandatan/cub-200-siglip-embeddings-distractor-set", SIGLIP_DIR, "siglip_image_embeddings_test.pkl"),
    (TEACHER_DATASET, TEACHER_ARTIFACT.parent, TEACHER_ARTIFACT.name),
]
for dataset, directory, sentinel in DOWNLOADS:
    if (directory / sentinel).exists():
        print(f"✓ {directory / sentinel}")
        continue
    directory.mkdir(parents=True, exist_ok=True)
    run("kaggle", "datasets", "download", "-d", dataset, "-p", directory, "--unzip")
    if not (directory / sentinel).exists():
        raise FileNotFoundError(f"{sentinel} missing from {dataset}")

# The scripts read SigLIP files and the image split from data/cub_200 off Kaggle.
data_dir = REPO_DIR / "data" / "cub_200"
data_dir.mkdir(parents=True, exist_ok=True)
links = {"image_split.json": SPLIT_DIR / "image_split.json"}
for name in ("siglip_text_embeddings.pkl", *(f"siglip_image_embeddings_{s}.pkl" for s in ("train", "val", "test"))):
    links[name] = SIGLIP_DIR / name
for name, target in links.items():
    if not target.exists():
        raise FileNotFoundError(target)
    link = data_dir / name
    if link.is_symlink() or link.exists():
        link.unlink()
    link.symlink_to(target)
    print(f"{link} -> {target}")
IMAGE_SPLIT_PATH = data_dir / "image_split.json"

In [ ]:
import torch

run("nvidia-smi")
NUM_GPUS = torch.cuda.device_count()
if NUM_GPUS == 0:
    raise RuntimeError("No CUDA GPU visible; the test evaluation needs at least one")
print(f"Idefics2 workers: {NUM_GPUS} (one per GPU)")

## Train M=30/50/100 rerankers

All three use the sweep-best M=30 recipe on the M=100 artifact; only
`data.max_candidates` changes. Retraining M=30 here keeps the recipe identical
across pool sizes. Each run writes to its own directory, and finished runs are
skipped.

In [ ]:
TRAIN_ARGS = [
    "data.target=margin", "data.batch_size=64", "experiment.seed=40",
    "model.architecture=interaction_mlp", "model.dropout=0.1", "model.hidden_dim=256",
    "objective.name=hybrid_listwise_pairwise", "objective.hybrid_listwise_weight=0.1",
    "objective.pairwise_min_target_gap=0.02", "objective.pairwise_score_temperature=1.0",
    "optimization.learning_rate=0.0003", "optimization.weight_decay=0.001",
    "optimization.epochs=100",
]


def best_checkpoint(pool_size):
    found = sorted((TRAIN_OUTPUT_ROOT / f"m{pool_size}").glob("*/best.pt"))
    if len(found) > 1:
        raise RuntimeError(f"Several runs under m{pool_size}; keep exactly one: {found}")
    return found[0] if found else None


for pool_size in POOL_SIZES:
    if best_checkpoint(pool_size):
        print(f"✓ M={pool_size} already trained: {best_checkpoint(pool_size)}")
        continue
    run(
        sys.executable, "-m", "scripts.train_reranker",
        f"data.artifact_path={TEACHER_ARTIFACT}", f"data.max_candidates={pool_size}",
        *TRAIN_ARGS, f"output.dir={TRAIN_OUTPUT_ROOT / f'm{pool_size}'}",
    )

In [ ]:
# Validation comparison; the primary selector is resolved from this table only.
import pandas as pd

rows = []
for pool_size in POOL_SIZES:
    checkpoint = torch.load(best_checkpoint(pool_size), map_location="cpu", weights_only=False)
    metrics = checkpoint["metrics"]
    rows.append({
        "selector": f"reranker_m{pool_size}",
        "best_epoch": checkpoint["epoch"],
        "val_selected_acc": metrics["restricted_selected_accuracy"],
        "val_pool_oracle_acc": metrics.get("restricted_pool_oracle_accuracy"),
        "val_mean_margin_regret": metrics.get("mean_margin_regret"),
    })
validation = pd.DataFrame(rows).set_index("selector")
display(validation)

if PRIMARY_SELECTOR == "auto":
    primary = validation["val_selected_acc"].idxmax()
elif PRIMARY_SELECTOR in validation.index:
    primary = PRIMARY_SELECTOR
else:
    raise ValueError(f"PRIMARY_SELECTOR must be 'auto' or one of {list(validation.index)}")
print(f"Primary selector: {primary} (validation accuracy {validation.loc[primary, 'val_selected_acc']:.4f})")

## Test evaluation

The primary selector printed above is passed to every run below.
The smoke test scores 20 queries locally to catch setup errors cheaply.

In [ ]:
EVAL_ARGS = [
    f"dataset.image_split_path={IMAGE_SPLIT_PATH}",
    *(f"selectors.reranker_m{m}.checkpoint={best_checkpoint(m)}" for m in POOL_SIZES),
    f"reranker.teacher_artifact={TEACHER_ARTIFACT}",
    f"primary_selector={primary}",
    f"hardware.num_gpus={NUM_GPUS}",
]

run(
    sys.executable, "-m", "scripts.evaluate_reranker_test", *EVAL_ARGS,
    "scoring.num_queries=20", "output.results_kaggle_dataset=null",
    f"output.save_dir={SMOKE_OUTPUT_ROOT}",
)

In [ ]:
# Full test run. Re-run this cell after an interruption to resume.
RESULT_DIR = EVAL_OUTPUT_ROOT / "reranker_test_eval"

resume = RESUME_FROM
if resume == "auto":
    # Newest result of any status; only a partial one is worth resuming.
    results_so_far = sorted(RESULT_DIR.glob("*.pkl"), key=lambda path: path.stat().st_mtime)
    resume = results_so_far[-1] if results_so_far and "_partial" in results_so_far[-1].name else None
print("Resuming from:", resume)

run(
    sys.executable, "-m", "scripts.evaluate_reranker_test", *EVAL_ARGS,
    f"output.save_dir={EVAL_OUTPUT_ROOT}",
    f"output.results_kaggle_dataset={RESULTS_KAGGLE_DATASET or 'null'}",
    f"limits.resume_from={resume or 'null'}",
)

In [ ]:
# Summary of the newest complete run.
import json

complete = [
    path for path in sorted(RESULT_DIR.glob("*.json"), key=lambda path: path.stat().st_mtime)
    if json.loads(path.read_text())["results"].get("complete")
]
summary = json.loads(complete[-1].read_text())
results = summary["results"]
print(complete[-1].name, f"— {results['completed_queries']} test queries, primary = {results['primary_selector']}")

display(pd.DataFrame({
    name: {key: condition[key] for key in ("accuracy", "top5_accuracy", "mean_per_class_accuracy", "mean_reciprocal_rank")}
    for name, condition in results["conditions"].items()
}).T)
display(pd.DataFrame(results["paired_vs_clip_top1"]).T[
    ["accuracy_difference", "ci_low", "ci_high", "mcnemar_exact_p", "treatment_only_correct", "control_only_correct"]
])
display(pd.DataFrame(results["selection"]).T)